In [1]:
# import pyautogui
# import pyscreeze

In [1]:
# img = pyscreeze.screenshot()
# type(img)
# img.save('screenshot.png')

In [15]:
import cv2
import numpy as np

# 加载模板图片和大图片
template = cv2.imread('recycle.png', 0)
large_image = cv2.imread('screenshot.png', 0)

# 使用模板匹配 (TM_CCOEFF_NORMED)
result = cv2.matchTemplate(large_image, template, cv2.TM_CCOEFF_NORMED)
min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)

# 获取匹配坐标
top_left = max_loc
h, w = template.shape
bottom_right = (top_left[0] + w, top_left[1] + h)

# 绘制边界框
output_image = cv2.cvtColor(large_image, cv2.COLOR_GRAY2BGR)
cv2.rectangle(output_image, top_left, bottom_right, (0, 255, 0), 2)

# 保存或显示结果
cv2.imwrite('output.png', output_image)



True

In [16]:
# import pyautogui
# pyautogui.locateOnScreen('recycle.png')

In [17]:
# from PIL import Image
# Image.open('output.png')

In [58]:
import os
import base64
import json
import pyautogui
 
from openai import OpenAI
 
client = OpenAI(
    api_key = os.getenv("OPENAI_API_KEY"), 
    base_url = os.getenv("OPENAI_API_BASE")
)
 
# 对图片进行base64编码
with open("screenshot.png", 'rb') as f:
    img_base = base64.b64encode(f.read()).decode('utf-8')

prompt = """
1. 这是一张屏幕的截图
2. 请返回“老鼠”图标的位置
3. 使用0-1间的数值来表示在屏幕中的位置 (0 <= 左,上,右,下 <= 1)
4. 除了坐标信息不返回其他任何信息。
5. 返回格式为: [[左,上], [右,下]]
"""
response = client.chat.completions.create(
    model="moonshot-v1-8k-vision-preview", 
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{img_base}"
                    }
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    ]
)
(x1, y1), (x2, y2) = json.loads(response.choices[0].message.content)
pyautogui.moveTo(round((x1 + x2) * 1920 / 2), round((y1 + y2) * 1080 / 2), 5)

In [60]:
from openai import OpenAI
 
client = OpenAI(
    api_key = os.getenv("OPENAI_API_KEY"), 
    base_url = os.getenv("OPENAI_API_BASE")
)
 
completion = client.chat.completions.create(
    model = "kimi-k2-0905-preview",
    messages = [
        {"role": "system", "content": "你是 Kimi，由 Moonshot AI 提供的人工智能助手，你更擅长中文和英文的对话。你会为用户提供安全，有帮助，准确的回答。同时，你会拒绝一切涉及恐怖主义，种族歧视，黄色暴力等问题的回答。Moonshot AI 为专有名词，不可翻译成其他语言。"},
        {"role": "user", "content": "编程判断 3214567 是否是素数。"}
    ],
    tools = [{
        "type": "function",
        "function": {
            "name": "CodeRunner",
            "description": "代码执行器，支持运行 python 和 javascript 代码",
            "parameters": {
                "properties": {
                    "language": {
                        "type": "string",
                        "enum": ["python", "javascript"]
                    },
                    "code": {
                        "type": "string",
                        "description": "代码写在这里"
                    }
                },
            "type": "object"
            }
        }
    }],
    temperature = 0.6,
)
 
print(completion.choices[0].message)

ChatCompletionMessage(content='我来帮你判断 3214567 是否是素数。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='CodeRunner:0', function=Function(arguments='{"language": "python", "code": "def is_prime(n):\\n    if n < 2:\\n        return False\\n    if n == 2:\\n        return True\\n    if n % 2 == 0:\\n        return False\\n    \\n    # 只需要检查到 sqrt(n)\\n    import math\\n    max_divisor = math.isqrt(n) + 1\\n    \\n    # 检查奇数因子\\n    for d in range(3, max_divisor, 2):\\n        if n % d == 0:\\n            return False\\n    \\n    return True\\n\\n# 测试 3214567\\nnumber = 3214567\\nresult = is_prime(number)\\n\\nprint(f\\"{number} 是素数: {result}\\")\\n\\n# 为了验证，我们也可以找出它的因子（如果不是素数）\\nif not result:\\n    def find_factors(n):\\n        factors = []\\n        d = 2\\n        while d * d <= n:\\n            while n % d == 0:\\n                factors.append(d)\\n                n //= d\\n            d += 1\\n 

In [66]:
result = dict(completion.choices[0].message.tool_calls[0])

In [70]:
func = result['function']

In [71]:
func.name

'CodeRunner'

In [76]:

args = eval(func.arguments)

In [80]:
print(args['language'])

python


In [81]:
print(args['code'])

def is_prime(n):
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    
    # 只需要检查到 sqrt(n)
    import math
    max_divisor = math.isqrt(n) + 1
    
    # 检查奇数因子
    for d in range(3, max_divisor, 2):
        if n % d == 0:
            return False
    
    return True

# 测试 3214567
number = 3214567
result = is_prime(number)

print(f"{number} 是素数: {result}")

# 为了验证，我们也可以找出它的因子（如果不是素数）
if not result:
    def find_factors(n):
        factors = []
        d = 2
        while d * d <= n:
            while n % d == 0:
                factors.append(d)
                n //= d
            d += 1
        if n > 1:
            factors.append(n)
        return factors
    
    factors = find_factors(number)
    print(f"{number} 的质因数分解: {factors}")
    print(f"验证: {' × '.join(map(str, factors))} = {eval('*'.join(map(str, factors)))}")
